<div align="center">

## **Vampirul**
### *Novel Analysis Pipeline — Prerequisites*

</div>

---

## Installation

Install all required dependencies with:

`pip install -r requirements.txt`

---

## Language Model Setup (One-Time Step)

Download the **Stanza Romanian language model**:

`python -c "import stanza; stanza.download('ro')"`

---

<div align="center">

───────────────────────────────────────────────  
   You’re now ready to run the Vampirul analysis pipeline    
───────────────────────────────────────────────  

</div>

## EXTRACT DIALOGUES

In [ ]:
import re
from docx import Document
import json
from datetime import datetime

def is_attribution_only(text):
    attribution_verbs = ['zise', 'spuse', 'răspunse', 'întrebă', 'strigă', 'șopti', 'murmură']
    words = text.lower().split()
    
    if len(words) <= 3 and any(verb in words for verb in attribution_verbs):
        return True
    return False

def extract_dialogue_with_dash(text):
    if not text.startswith('—'):
        return None
    
    text = text[1:].strip()
    
    if is_attribution_only(text):
        return None
    
    match = re.match(r'^([^.!?]+[.!?])', text)
    if match:
        dialogue = match.group(1).strip()
        if len(dialogue.split()) >= 3:
            return dialogue
    
    return None

def extract_dialogue_with_quotes(text):
    pattern = r'„([^"]+)"'
    matches = re.findall(pattern, text)
    
    dialogues = []
    for match in matches:
        cleaned = re.sub(r'—[^—]+—', '', match).strip()
        
        if len(cleaned.split()) >= 3:
            dialogues.append(cleaned)
    
    return dialogues

def extract_all_dialogues_smart(docx_path, output_json='dialogues_extracted.json'):

    doc = Document(docx_path)
    
    all_dialogues = []
    paragraph_index = 0
    
    for para in doc.paragraphs:
        text = para.text.strip()
        
        if not text:
            continue
        
        if text.startswith('—'):
            dialogue = extract_dialogue_with_dash(text)
            if dialogue:
                all_dialogues.append({
                    'id': len(all_dialogues) + 1,
                    'type': 'dash',
                    'paragraph_index': paragraph_index,
                    'text': dialogue,
                    'word_count': len(dialogue.split()),
                    'speaker_name': None,
                    'dialogue_partner': None,
                    'sentiment': None,
                    'location': None,
                    'silent_participants': []
                })
        
        if '„' in text:
            dialogues = extract_dialogue_with_quotes(text)
            for dialogue in dialogues:
                all_dialogues.append({
                    'id': len(all_dialogues) + 1,
                    'type': 'quotes',
                    'paragraph_index': paragraph_index,
                    'text': dialogue,
                    'word_count': len(dialogue.split()),
                    'speaker_name': None,
                    'dialogue_partner': None,
                    'sentiment': None,
                    'location': None,
                    'silent_participants': []
                })
        
        paragraph_index += 1
    
    output = {
        'Total_dialogues': len(all_dialogues),
        'by_type': {
            'dash': len([d for d in all_dialogues if d['type'] == 'dash']),
            'quotes': len([d for d in all_dialogues if d['type'] == 'quotes'])
        },
        'dialogues': all_dialogues
    }
    
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    
    print(f"Total dialogues: {len(all_dialogues)}")
    print(f"   With '—': {output['by_type']['dash']}")
    print(f"   With '„...”': {output['by_type']['quotes']}")
    print(f"Saved to {output_json}")
    
    return all_dialogues

dialogues = extract_all_dialogues_smart('vampirul_clean.docx')

def generate_validation_state(dialogues, output_file='dialogues_validation_state.json'):
    state_dialogues = []
    for d in dialogues:
        state_dialogues.append({
            'id':                  d['id'],
            'status':              'pending',
            'original_text':       d['text'],
            'current_text':        d['text'],
            'word_count':          d['word_count'],
            'validated_at':        None,
            'notes':               '',
            'was_edited':          False,
            'speaker_name':        d.get('speaker_name'),
            'dialogue_partner':    d.get('dialogue_partner'),
            'sentiment':           d.get('sentiment'),
            'location':            d.get('location'),
            'silent_participants': d.get('silent_participants', []),
        })

    state = {
        'session': {
            'created_at':      datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'last_updated':    datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'total_dialogues': len(state_dialogues),
            'stats': {
                'validated': 0,
                'edited':    0,
                'deleted':   0,
                'pending':   len(state_dialogues),
            }
        },
        'dialogues': state_dialogues,
    }

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

    print(f"Validation state file saved → {output_file}")
    print(f"{len(state_dialogues)} dialogues ready for validation")

generate_validation_state(dialogues)

## WEBSERVER FOR FILE READ/WRITE FUNCTIONALITY DIALOGUE VALIDATOR

In [ ]:
from flask import Flask, jsonify, request, send_file
import json
import os
from datetime import datetime
import threading

app = Flask(__name__)

@app.after_request
def after_request(response):
    response.headers.add('Access-Control-Allow-Origin', '*')
    response.headers.add('Access-Control-Allow-Headers', 'Content-Type,Authorization')
    response.headers.add('Access-Control-Allow-Methods', 'GET,POST,DELETE,OPTIONS,PUT')
    return response

STATE_FILE = 'dialogues_validation_state.json'

@app.route('/api/state', methods=['GET', 'OPTIONS'])
def get_state():
    if request.method == 'OPTIONS':
        return '', 204
    
    if not os.path.exists(STATE_FILE):
        return jsonify({'error': 'State file not found'}), 404
    
    with open(STATE_FILE, 'r', encoding='utf-8') as f:
        state = json.load(f)
    
    return jsonify(state)


@app.route('/api/state', methods=['POST', 'OPTIONS'])
def save_state():
    if request.method == 'OPTIONS':
        return '', 204
    
    try:
        state = request.json
        state['session']['last_updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        with open(STATE_FILE, 'w', encoding='utf-8') as f:
            json.dump(state, f, ensure_ascii=False, indent=2)
        
        return jsonify({'success': True, 'message': 'State saved'})
    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/api/dialogue/add', methods=['POST', 'OPTIONS'])
def add_dialogue():
    if request.method == 'OPTIONS':
        return '', 204
    
    try:
        data = request.json
        insert_after_id = data.get('after_id')
        new_text = data.get('text', '').strip()
        
        if not new_text:
            return jsonify({'error': 'Text cannot be empty'}), 400
        
        with open(STATE_FILE, 'r', encoding='utf-8') as f:
            state = json.load(f)
        
        insert_index = 0
        if insert_after_id:
            for i, d in enumerate(state['dialogues']):
                if d['id'] == insert_after_id:
                    insert_index = i + 1
                    break
        
        new_dialogue = {
            'id': None,
            'status': 'edited',
            'original_text': new_text,
            'current_text': new_text,
            'word_count': len(new_text.split()),
            'validated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'notes': 'Manually added',
            'was_edited': True,
            'speaker_name': None,
            'dialogue_partner': None,
            'sentiment': None,
            'location': None,
            'silent_participants': []
        }
        
        state['dialogues'].insert(insert_index, new_dialogue)
        
        for i, d in enumerate(state['dialogues'], 1):
            d['id'] = i
        
        state['session']['total_dialogues'] = len(state['dialogues'])
        state['session']['stats']['edited'] = sum(1 for d in state['dialogues'] if d['status'] == 'edited')
        state['session']['stats']['pending'] = sum(1 for d in state['dialogues'] if d['status'] == 'pending')
        
        with open(STATE_FILE, 'w', encoding='utf-8') as f:
            json.dump(state, f, ensure_ascii=False, indent=2)
        
        return jsonify({'success': True, 'new_id': new_dialogue['id'], 'message': 'Dialogue added'})
    
    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/api/dialogue/delete/<int:dialogue_id>', methods=['DELETE', 'OPTIONS'])
def delete_dialogue(dialogue_id):
    if request.method == 'OPTIONS':
        return '', 204
    
    try:
        with open(STATE_FILE, 'r', encoding='utf-8') as f:
            state = json.load(f)
        
        original_count = len(state['dialogues'])
        state['dialogues'] = [d for d in state['dialogues'] if d['id'] != dialogue_id]
        
        if len(state['dialogues']) == original_count:
            return jsonify({'error': 'Dialogue not found'}), 404
        
        for i, d in enumerate(state['dialogues'], 1):
            d['id'] = i
        
        state['session']['total_dialogues'] = len(state['dialogues'])
        state['session']['stats'] = {
            'validated': sum(1 for d in state['dialogues'] if d['status'] == 'validated'),
            'edited': sum(1 for d in state['dialogues'] if d['status'] == 'edited'),
            'deleted': sum(1 for d in state['dialogues'] if d['status'] == 'deleted'),
            'pending': sum(1 for d in state['dialogues'] if d['status'] == 'pending')
        }
        
        with open(STATE_FILE, 'w', encoding='utf-8') as f:
            json.dump(state, f, ensure_ascii=False, indent=2)
        
        return jsonify({'success': True, 'message': f'Dialogue {dialogue_id} deleted'})
    
    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/')
def serve_html():
    return send_file('validation_interface.html')

def run_server():
    app.run(host='localhost', port=5000, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

def generate_work_file(state_file='dialogues_validation_state.json',
                        output_file='dialogue_id_char_work.json'):
    if not os.path.exists(state_file):
        print(f"   {state_file} not found — run cell 1 first")
        return

    with open(state_file, 'r', encoding='utf-8') as f:
        state = json.load(f)

    dialogues = []
    for d in state.get('dialogues', []):
        dialogues.append({
            'id':                  d['id'],
            'status':              'pending',
            'text':                d.get('current_text') or d.get('original_text', ''),
            'word_count':          d.get('word_count', 0),
            'speaker_name':        d.get('speaker_name'),
            'dialogue_partner':    d.get('dialogue_partner'),
            'silent_participants': d.get('silent_participants', []),
            'sentiment':           d.get('sentiment'),
            'location':            d.get('location'),
        })

    work = {'dialogues': dialogues}

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(work, f, ensure_ascii=False, indent=2)

    print(f"   Work file saved → {output_file}")
    print(f"   {len(dialogues)} dialogues ready for character assignment")

generate_work_file()

print("="*60)
print("SERVER STARTED!")
print("="*60)
print("URL: http://localhost:5000")
print("API: http://localhost:5000/api/state")
print("\nServer is running in background")
print("Open http://localhost:5000 in your browser")
print("\nTo stop: restart Jupyter kernel")
print("="*60)

## CHARACTER IDENTIFICATION INTERFACE

In [ ]:
from flask import Flask, jsonify, request, send_file
import json
import os
from datetime import datetime
import threading

app = Flask(__name__)

@app.after_request
def after_request(response):
    response.headers.add('Access-Control-Allow-Origin', '*')
    response.headers.add('Access-Control-Allow-Headers', 'Content-Type,Authorization')
    response.headers.add('Access-Control-Allow-Methods', 'GET,POST,PUT,OPTIONS')
    return response

WORK_FILE = 'dialogue_id_char_work.json'
SESSION_FILE = 'character_assignment_session.json'

DEFAULT_CHARACTERS = [
    "Corneliu Cociu",
    "Contesa Ermina de Kolovrat",
    "Baronul Julius-Adalbert Wirth",
    "Rudy Tamasch",
    "Eva Tamasch",
    "Preotul Almaș Becker",
    "Vampirul",
    "Geza Vendelin",
    "Ajutorul de primar",
    "Bela Töröky",
    "Solomon Bastius",
    "Moș Miloș Kiraylli",
    "Cârciumarul din Steierdorf",
    "Madam Elsa",
    "Madam lulișca",
    "Fritz Krauss",
    "Merza Tanady",
    "Moș Badea Hâncu",
    "Nagy Papian",
    "Mitică Șerbănescu-Vaslui",
    "Ludovic Vecserdi",
    "Marcu",
    "Magistratul Matei",
    "Toni Mexelkiș",
    "Alfons Kleipert",
    "Moise Țepeneag",
    "Horațiu"
]

@app.route('/api/data', methods=['GET', 'OPTIONS'])
def get_data():
    if request.method == 'OPTIONS':
        return '', 204
    
    if not os.path.exists(WORK_FILE):
        return jsonify({'error': 'Work file not found'}), 404
    
    with open(WORK_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_characters = set(DEFAULT_CHARACTERS)
    
    for dialogue in data.get('dialogues', []):
        if dialogue.get('speaker_name'):
            all_characters.add(dialogue['speaker_name'])
        if dialogue.get('dialogue_partner'):
            all_characters.add(dialogue['dialogue_partner'])
        for participant in dialogue.get('silent_participants', []):
            if participant:
                all_characters.add(participant)
    
    data['available_characters'] = sorted(list(all_characters))
    
    session = {}
    if os.path.exists(SESSION_FILE):
        with open(SESSION_FILE, 'r', encoding='utf-8') as f:
            session = json.load(f)
    else:
        session = {
            'last_dialogue_id': None,
            'last_filter': 'all',
            'started_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'last_updated': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        with open(SESSION_FILE, 'w', encoding='utf-8') as f:
            json.dump(session, f, ensure_ascii=False, indent=2)
    
    data['session'] = session
    
    return jsonify(data)

@app.route('/api/dialogue/<int:dialogue_id>', methods=['PUT', 'OPTIONS'])
def update_dialogue(dialogue_id):
    if request.method == 'OPTIONS':
        return '', 204
    
    try:
        update_data = request.json
        print(f"[DIALOGUE] updating dialogue ID: {dialogue_id}")
        print(f"[DIALOGUE] data: {update_data}")
        
        with open(WORK_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        dialogue = None
        for d in data['dialogues']:
            if d['id'] == dialogue_id:
                dialogue = d
                break
        
        if not dialogue:
            return jsonify({'error': 'Dialogue not found'}), 404
        
        dialogue['speaker_name'] = update_data.get('speaker_name')
        dialogue['dialogue_partner'] = update_data.get('dialogue_partner')
        dialogue['silent_participants'] = update_data.get('silent_participants', [])
        
        dialogue['status'] = 'characters_set'
        print(f"[DIALOGUE] status set to: characters_set")
        
        dialogue['character_updated_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        with open(WORK_FILE, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        
        print(f"[DIALOGUE] saved to {WORK_FILE}")
        
        if os.path.exists(SESSION_FILE):
            with open(SESSION_FILE, 'r', encoding='utf-8') as f:
                session = json.load(f)
        else:
            session = {}
        
        next_id = dialogue_id + 1
        session['last_dialogue_id'] = next_id
        session['last_updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        with open(SESSION_FILE, 'w', encoding='utf-8') as f:
            json.dump(session, f, ensure_ascii=False, indent=2)
        
        print(f"[DIALOGUE] session saved. Next dialogue ID: {next_id}")
        
        return jsonify({'success': True, 'message': 'Dialogue updated', 'next_dialogue_id': next_id})
    
    except Exception as e:
        print(f"[DIALOGUE ERROR] {str(e)}")
        return jsonify({'error': str(e)}), 500

@app.route('/api/character/add', methods=['POST', 'OPTIONS'])
def add_character():
    if request.method == 'OPTIONS':
        return '', 204
    
    try:
        new_character = request.json.get('name', '').strip()
        
        if not new_character:
            return jsonify({'error': 'Character name cannot be empty'}), 400
        
        return jsonify({'success': True, 'character': new_character})
    
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/')
def serve_html():
    return send_file('character_assignment.html')

def run_server():
    app.run(host='localhost', port=5001, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("="*60)
print("CHARACTER ASSIGNMENT SERVER STARTED!")
print("="*60)
print("URL: http://localhost:5001")
print("API: http://localhost:5001/api/data")
print("\nServer is running in background")
print("Open http://localhost:5001 in your browser")
print("\nTo stop: restart Jupyter kernel")
print("="*60)

## CLEAN FINAL JSON WITH DIALOGUE PARTNERS

In [ ]:
import json
from datetime import datetime

def clean_dialogues(input_file, output_file):
    
    print(f"Reading {input_file}...")
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    original_count = len(data.get('dialogues', []))
    print(f"Loaded {original_count} dialogues")
    
    keep_fields = [
        'id',
        'text', 
        'word_count',
        'speaker_name',
        'dialogue_partner',
        'silent_participants'
    ]
    
    cleaned_dialogues = []
    
    for new_id, dialogue in enumerate(data.get('dialogues', []), start=1):
        cleaned_dialogue = {}
        
        cleaned_dialogue['id'] = new_id
        
        for field in keep_fields:
            if field == 'id':
                continue  
            cleaned_dialogue[field] = dialogue.get(field, None)
        
        cleaned_dialogues.append(cleaned_dialogue)
    
    print(f"Saving to {output_file}...")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cleaned_dialogues, f, ensure_ascii=False, indent=2)
    
    print(f"Done! {len(cleaned_dialogues)} clean and reindexed dialogues saved to {output_file}")
    
    print("\nStatistics:")
    stats = {
        'Total dialogues': len(cleaned_dialogues),
        'With speaker': sum(1 for d in cleaned_dialogues if d.get('speaker_name')),
        'With partner': sum(1 for d in cleaned_dialogues if d.get('dialogue_partner')),
        'With silent participants': sum(1 for d in cleaned_dialogues 
                                        if d.get('silent_participants') and len(d['silent_participants']) > 0)
    }
    
    for key, value in stats.items():
        print(f"  - {key}: {value}")
    
    print("\nReindexing check:")
    print(f"  - First ID: {cleaned_dialogues[0]['id']}")
    print(f"  - Last ID: {cleaned_dialogues[-1]['id']}")
    print(f"  - Total dialogues: {len(cleaned_dialogues)}")
    
    return cleaned_dialogues

if __name__ == '__main__':
    input_file = 'dialogue_id_char_work.json'
    output_file = 'dialogues_with_participants.json'
    
    print("="*60)
    print("Dialogues cleaning...")
    print("="*60)
    
    cleaned_data = clean_dialogues(input_file, output_file)
    
    print("\nProcessing finished!")
    print(f"Output file: {output_file}")
    print("="*60)

## EXTRACT CHARACTERS FOR GENDER ASSIGNMENT

In [ ]:
import json
from collections import Counter

def extract_characters(dialogues_file):
    
    print("Reading dialogues...")
    with open(dialogues_file, 'r', encoding='utf-8') as f:
        dialogues = json.load(f)
    
    characters = set()
    character_stats = Counter()
    
    for dialogue in dialogues:
        speaker = dialogue.get('speaker_name')
        partner = dialogue.get('dialogue_partner')
        silent = dialogue.get('silent_participants', [])
        
        if speaker:
            characters.add(speaker)
            character_stats[speaker] += 1
        
        if partner:
            characters.add(partner)
        
        for person in silent:
            if person:
                characters.add(person)
    
    characters = sorted(list(characters))
    
    print(f"{len(characters)} Characters found")
    
    metadata = {
        "characters": []
    }
    
    for char in characters:
        metadata["characters"].append({
            "name": char,
            "gender": None, 
            "dialogue_count": character_stats.get(char, 0),
            "notes": ""
        })
    
    return metadata

def save_metadata(metadata, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print(f"Metadata saved to {output_file}")

def main():
    print("="*70)
    print("CHARACTER METADATA EXTRACTION")
    print("="*70)
    
    dialogues_file = 'dialogues_with_participants.json'
    output_file = 'characters_metadata.json'
    
    metadata = extract_characters(dialogues_file)
    
    save_metadata(metadata, output_file)
    
    print("\nDone!")
    print(f"Next step: Open character_gender_editor.html in the browser to complete the gender information for each character.")
    print("="*70)

if __name__ == '__main__':
    main()

## GENERATE CHARACTER NETWORK BASED ON JSON DIALOGUES ARRAY

### CONFIG & CONSTANTS

In [ ]:
import json
import re
import os
import networkx as nx
import stanza
from collections import defaultdict, Counter
from datetime import datetime

DIALOGUES_FILE    = "dialogues_with_participants.json"
METADATA_FILE     = "characters_metadata.json"
OUTPUT_FILE       = "network_data.json" 

# Vocabulary thresholds
RELIABILITY_LOW    = 100
RELIABILITY_MEDIUM = 400
TOP_N_WORDS        = 20
MIN_PAIR_WORDS     = 30
CONTENT_POS        = {"NOUN", "VERB", "ADJ", "ADV"}

RO_STOPWORDS = {
    "eu", "tu", "el", "ea", "noi", "voi", "ei", "ele",
    "meu", "mea", "tău", "ta", "său", "sa", "nostru", "noastră", "vostru", "voastră",
    "îmi", "îți", "îi", "ne", "vă", "le", "mi", "ți", "si",
    "mine", "tine", "sine", "nouă", "vouă", "lor",
    "acesta", "aceasta", "acela", "aceea", "același", "aceeași",
    "care", "ce", "cine", "cui", "căruia", "căreia", "cărora",
    "în", "la", "de", "pe", "cu", "din", "prin", "pentru", "despre",
    "după", "înainte", "înapoi", "între", "dintre", "asupra", "înaintea",
    "că", "și", "sau", "dar", "ci", "însă", "deci", "totuși", "dacă",
    "când", "unde", "cum", "cât", "deși", "fie", "nici", "ori",
    "un", "o", "cel", "cea", "cei", "cele", "al", "a", "ai", "ale",
    "fi", "avea", "a", "mai", "tot", "foarte", "deja", "doar", "chiar",
    "nu", "da", "aici", "acolo", "acum", "atunci", "apoi", "deci",
    "bine", "rău", "mult", "puțin", "astfel", "toată", "toate",
    "domnule", "doamne", "vai", "ei", "ha", "ah", "oh",
}

print("Config loaded.")
print(f"   Dialogues : {DIALOGUES_FILE}")
print(f"   Metadata  : {METADATA_FILE}")
print(f"   Output    : {OUTPUT_FILE}")


### LOAD DATA

In [ ]:
def load_dialogues(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_character_metadata(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
        metadata = {}
        for char in data.get('characters', []):
            metadata[char['name']] = {
                'gender':        char.get('gender'),
                'notes':         char.get('notes', ''),
                'meta_identity': char.get('meta_identity'),
                'identity_type': char.get('identity_type'),
            }
        return metadata
    except FileNotFoundError:
        print(f"{filename} not found — gender info will not be included")
        return {}

dialogues         = load_dialogues(DIALOGUES_FILE)
character_metadata = load_character_metadata(METADATA_FILE)

print(f"{len(dialogues)} dialogues loaded from '{DIALOGUES_FILE}'")
print(f"{len(character_metadata)} characters loaded from '{METADATA_FILE}'" if character_metadata
        else "No metadata loaded — run gender assignment cell first if needed")


### BUILD NETWORK GRAPH

In [ ]:
def build_network(dialogues):
    G = nx.Graph()

    direct_interactions     = defaultdict(int)
    copresence_interactions = defaultdict(int)
    character_dialogues     = defaultdict(int)
    speaker_words           = defaultdict(int)
    edge_words              = defaultdict(int)

    for dialogue in dialogues:
        speaker    = dialogue.get('speaker_name')
        partner    = dialogue.get('dialogue_partner')
        silent     = dialogue.get('silent_participants', [])
        word_count = dialogue.get('word_count', 0)

        if speaker:
            character_dialogues[speaker] += 1
            speaker_words[speaker] += word_count
            if speaker not in G:
                G.add_node(speaker)

        if partner and partner not in G:
            G.add_node(partner)

        if speaker and partner and speaker != partner:
            pair = tuple(sorted([speaker, partner]))
            direct_interactions[pair] += 1
            edge_words[(speaker, partner)] += word_count

        all_present = []
        if speaker: all_present.append(speaker)
        if partner and partner != speaker: all_present.append(partner)
        all_present.extend(silent)
        for person in silent:
            if person not in G:
                G.add_node(person)
        for i, p1 in enumerate(all_present):
            for p2 in all_present[i+1:]:
                if p1 != p2:
                    pair = tuple(sorted([p1, p2]))
                    copresence_interactions[pair] += 1

    all_pairs = set(direct_interactions) | set(copresence_interactions)
    for pair in all_pairs:
        p1, p2 = pair
        words_p1 = edge_words.get((p1, p2), 0)
        words_p2 = edge_words.get((p2, p1), 0)
        G.add_edge(p1, p2,
            direct     = direct_interactions.get(pair, 0),
            copresence = copresence_interactions.get(pair, 0),
            words      = words_p1 + words_p2,
        )

    for u, v in G.edges():
        d = G[u][v].get('direct', 0)
        c = G[u][v].get('copresence', 0)
        G[u][v]['weight'] = d * 2 + c
        if 'words' not in G[u][v]:
            G[u][v]['words'] = 0

    for node in G.nodes():
        G.nodes[node]['dialogue_count'] = character_dialogues.get(node, 0)
        G.nodes[node]['degree']         = G.degree(node)
        G.nodes[node]['speaker_words']  = speaker_words.get(node, 0)

    print(f"{G.number_of_nodes()} Characters | {G.number_of_edges()} Edges")
    print(f"{len(direct_interactions)} Direct pairs | {len(copresence_interactions)} Co-presence pairs")
    return G, edge_words

G, edge_words = build_network(dialogues)

### CALCULATE METRICS & COMMUNITIES

In [ ]:
def calculate_metrics(G):
    for u, v in G.edges():
        G[u][v]['inv_weight'] = 1.0 / (G[u][v].get('weight', 1) + 1)

    degree_cent      = nx.degree_centrality(G)
    print("   Computing betweenness centrality...")
    betweenness_cent = nx.betweenness_centrality(G, weight='inv_weight')
    print("   Computing closeness centrality...")
    closeness_cent   = nx.closeness_centrality(G)
    print("   Computing eigenvector centrality...")
    try:
        eigenvector_cent = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
    except:
        eigenvector_cent = {n: 0 for n in G.nodes()}

    print("   Detecting communities...")
    from networkx.algorithms import community
    communities_sets = community.greedy_modularity_communities(G, weight='weight')
    communities = {}
    for i, comm in enumerate(communities_sets):
        for node in comm:
            communities[node] = i

    for node in G.nodes():
        G.nodes[node]['degree_centrality']      = degree_cent.get(node, 0)
        G.nodes[node]['betweenness_centrality'] = betweenness_cent.get(node, 0)
        G.nodes[node]['closeness_centrality']   = closeness_cent.get(node, 0)
        G.nodes[node]['eigenvector_centrality'] = eigenvector_cent.get(node, 0)
        G.nodes[node]['community']              = communities.get(node, 0)

    top_degree      = sorted(degree_cent.items(),      key=lambda x: x[1], reverse=True)[:10]
    top_betweenness = sorted(betweenness_cent.items(), key=lambda x: x[1], reverse=True)[:10]

    print(f"\nTop 10 by degree centrality:")
    for name, score in top_degree:
        print(f"   {name}: {score:.3f}")
    print(f"\nTop 10 by betweenness (bridges):")
    for name, score in top_betweenness:
        print(f"   {name}: {score:.3f}")
    print(f"\n{len(set(communities.values()))} communities detected")

    return communities

print("Calculating network metrics...")
communities = calculate_metrics(G)


### ANALYZE CHARACTER CONNECTIONS

In [ ]:
def analyze_character_connections(G, character_metadata):
    connections_data = {}

    for node in G.nodes():
        neighbors = list(G.neighbors(node))
        partners_data   = []
        gender_dist     = {'M': 0, 'F': 0, 'C': 0, 'Unknown': 0}
        speaker_total   = 0

        for neighbor in neighbors:
            e           = G[node][neighbor]
            direct      = e.get('direct', 0)
            copresence  = e.get('copresence', 0)
            words       = e.get('words', 0)
            weight      = e.get('weight', 0)
            gender      = character_metadata.get(neighbor, {}).get('gender')

            gender_dist[gender if gender else 'Unknown'] += 1

            partners_data.append({
                'name':             neighbor,
                'gender':           gender,
                'direct_dialogues': direct,
                'copresence':       copresence,
                'total_words':      words,
                'weight':           weight,
            })

        partners_data.sort(key=lambda x: x['weight'], reverse=True)

        total_partners = len(partners_data)
        gender_pct = {
            g: {'count': c, 'percentage': round(c / total_partners * 100, 1) if total_partners else 0}
            for g, c in gender_dist.items()
        }

        node_community    = G.nodes[node].get('community', 0)
        community_members = [n for n in G.nodes()
                            if G.nodes[n].get('community', -1) == node_community and n != node]

        speaker_words_total = G.nodes[node].get('speaker_words', 0)

        connections_data[node] = {
            'partners':              partners_data,
            'top_5_partners':        partners_data[:5],
            'gender_distribution':   gender_dist,
            'gender_percentages':    gender_pct,
            'total_words_exchanged': speaker_words_total,
            'total_connections':     total_partners,
            'community_members':     community_members[:10],
            'community_id':          node_community,
        }

    print(f"Connections analyzed for {len(connections_data)} characters")
    return connections_data

connections_analysis = analyze_character_connections(G, character_metadata)


## VOCABULARY ANALYSIS

### RUN STANZA NLP PIPELINE

In [ ]:
def run_stanza(dialogues):
    print("=" * 60)
    print("Loading Stanza Romanian pipeline...")
    print("=" * 60)
    nlp = stanza.Pipeline("ro", processors="tokenize,pos,lemma", verbose=False)

    grouped = defaultdict(list)
    for d in dialogues:
        spk = d.get("speaker_name")
        txt = d.get("text", "").strip()
        prt = d.get("dialogue_partner")
        if spk and txt:
            grouped[(spk, prt)].append(txt)

    processed = defaultdict(lambda: defaultdict(lambda: {
        "tokens": [], "lemmas": [], "pos": [], "content_lemmas": []
    }))

    total = len(grouped)
    print(f"Processing {total} speaker-partner pairs...\n")

    for idx, ((spk, prt), texts) in enumerate(grouped.items(), 1):
        doc = nlp(" ".join(texts))
        tokens, lemmas, pos_tags = [], [], []

        for sentence in doc.sentences:
            for word in sentence.words:
                tok = word.text.lower()
                lem = (word.lemma or tok).lower()
                pos = word.upos
                if re.fullmatch(r'[^\w\s]+', tok):
                    continue
                tokens.append(tok)
                lemmas.append(lem)
                pos_tags.append(pos)

        content_lemmas = [
            lem for lem, pos in zip(lemmas, pos_tags)
            if pos in CONTENT_POS and lem not in RO_STOPWORDS and len(lem) > 2
        ]

        entry = processed[spk][prt]
        entry["tokens"].extend(tokens)
        entry["lemmas"].extend(lemmas)
        entry["pos"].extend(pos_tags)
        entry["content_lemmas"].extend(content_lemmas)

        if idx % 20 == 0 or idx == total:
            print(f"  [{idx}/{total}] done")

    return processed

processed = run_stanza(dialogues)
print("\nStanza processing complete — `processed` ready in memory")


### BUILD VOCABULARY PROFILES

In [ ]:
def reliability_flag(n):
    if n < RELIABILITY_LOW:    return "low"
    if n < RELIABILITY_MEDIUM: return "medium"
    return "high"

def compute_ttr(tokens):
    if not tokens: return 0.0
    return round(len(set(tokens)) / len(tokens), 4)

def compute_mtld(tokens, threshold=0.72):
    if len(tokens) < 10: return 0.0
    def _pass(toks):
        factor_count, token_count, types = 0.0, 0, set()
        for tok in toks:
            token_count += 1
            types.add(tok)
            if len(types) / token_count <= threshold:
                factor_count += 1
                token_count, types = 0, set()
        if token_count > 0:
            ttr = len(types) / token_count
            if ttr < 1.0:
                factor_count += (1.0 - ttr) / (1.0 - threshold)
        return factor_count if factor_count > 0 else 1.0
    fwd = len(tokens) / _pass(tokens)
    bwd = len(tokens) / _pass(list(reversed(tokens)))
    return round((fwd + bwd) / 2, 2)

def jaccard(a, b):
    if not a or not b: return 0.0
    return round(len(a & b) / len(a | b), 4)

def avg_word_length(tokens):
    if not tokens: return 0.0
    return round(sum(len(t) for t in tokens) / len(tokens), 2)

def pos_distribution(pos_list):
    if not pos_list: return {}
    total  = len(pos_list)
    counts = Counter(pos_list)
    return {p: round(c / total * 100, 1) for p, c in counts.most_common()}

def spot_check(tokens, lemmas, n=25):
    pairs = [(t, l) for t, l in zip(tokens, lemmas) if t != l]
    if not pairs: return []
    step = max(1, len(pairs) // n)
    return [{"surface": t, "lemma": l} for t, l in pairs[::step][:n]]

def build_character_profiles(processed):
    char_all        = defaultdict(lambda: {"tokens": [], "lemmas": [], "pos": [], "content_lemmas": []})
    char_per_partner = defaultdict(dict)

    for spk, partner_data in processed.items():
        for prt, data in partner_data.items():
            for key in ("tokens", "lemmas", "pos", "content_lemmas"):
                char_all[spk][key].extend(data[key])
            char_per_partner[spk][prt] = data

    profiles = {}
    for char, agg in char_all.items():
        tokens         = agg["tokens"]
        lemmas         = agg["lemmas"]
        pos_tags       = agg["pos"]
        content_lemmas = agg["content_lemmas"]
        freq           = Counter(content_lemmas)

        profiles[char] = {
            "total_words":      len(tokens),
            "unique_lemmas":    len(set(lemmas)),
            "ttr":              compute_ttr(lemmas),
            "mtld":             compute_mtld(lemmas),
            "avg_word_length":  avg_word_length(tokens),
            "data_reliability": reliability_flag(len(tokens)),
            "top_words":        [{"word": w, "count": c} for w, c in freq.most_common(TOP_N_WORDS)],
            "pos_distribution": pos_distribution(pos_tags),
            "lemma_spot_check": spot_check(tokens, lemmas),
            "vocabulary_per_relationship": {},
        }

    for char, partner_data in char_per_partner.items():
        if char not in profiles: continue
        global_content = set(char_all[char]["content_lemmas"])
        rel_profiles   = {}

        for partner, data in partner_data.items():
            pair_words = len(data["tokens"])
            if pair_words < MIN_PAIR_WORDS:
                rel_profiles[partner] = {
                    "total_words": pair_words,
                    "data_note":   f"below {MIN_PAIR_WORDS} word threshold — metrics unreliable",
                    "top_words":   [], "vocab_overlap_with_global": None, "exclusive_words": [],
                }
                continue
            pair_content = data["content_lemmas"]
            pair_set     = set(pair_content)
            exclusive    = pair_set - (global_content - pair_set)

            rel_profiles[partner] = {
                "total_words":               pair_words,
                "unique_lemmas":             len(set(data["lemmas"])),
                "ttr":                       compute_ttr(data["lemmas"]),
                "avg_word_length":           avg_word_length(data["tokens"]),
                "top_words":                 [{"word": w, "count": c}
                                                for w, c in Counter(pair_content).most_common(TOP_N_WORDS)],
                "vocab_overlap_with_global": jaccard(pair_set, global_content),
                "exclusive_words":           sorted(exclusive)[:20],
            }

        profiles[char]["vocabulary_per_relationship"] = rel_profiles

    return profiles

profiles = build_character_profiles(processed)
print(f"Vocabulary profiles built for {len(profiles)} characters\n")

print("───────────────────────── Vocabulary Richness (MTLD) ───────────────────────────────")
ranked = sorted(profiles.items(), key=lambda x: x[1]["mtld"], reverse=True)
for char, p in ranked:
    flag = "!" if p["data_reliability"] == "low" else (" ~ " if p["data_reliability"] == "medium" else "   ")
    print(f"  {flag}{char:<45} MTLD={p['mtld']:>7.2f}  words={p['total_words']:>5}  [{p['data_reliability']}]")


### ENRICH AND EXPORT FINAL JSON

In [ ]:
def enrich_and_export(G, character_metadata, connections_analysis, profiles, output_file):
    protected = {os.path.abspath(DIALOGUES_FILE), os.path.abspath(METADATA_FILE)}
    if os.path.abspath(output_file) in protected:
        raise ValueError(f"OUTPUT_FILE '{output_file}' matches an input file. Aborting.")

    nodes = []
    for node in G.nodes():
        char = node
        nd = {
            'id':                     char,
            'label':                  char,
            'dialogue_count':         G.nodes[node].get('dialogue_count', 0),
            'degree':                 G.nodes[node].get('degree', 0),
            'degree_centrality':      G.nodes[node].get('degree_centrality', 0),
            'betweenness_centrality': G.nodes[node].get('betweenness_centrality', 0),
            'closeness_centrality':   G.nodes[node].get('closeness_centrality', 0),
            'eigenvector_centrality': G.nodes[node].get('eigenvector_centrality', 0),
            'community':              G.nodes[node].get('community', 0),
            'gender':                 character_metadata.get(char, {}).get('gender'),
            'notes':                  character_metadata.get(char, {}).get('notes', ''),
            'meta_identity':          character_metadata.get(char, {}).get('meta_identity'),
            'identity_type':          character_metadata.get(char, {}).get('identity_type'),
        }

        if char in connections_analysis:
            ca = connections_analysis[char]
            for partner_entry in ca.get('partners', []):
                pname    = partner_entry['name']
                rel_data = profiles.get(char, {}).get('vocabulary_per_relationship', {}).get(pname, {})
                partner_entry['vocab_overlap_score']        = rel_data.get('vocab_overlap_with_global')
                partner_entry['exclusive_words_to_partner'] = rel_data.get('exclusive_words', [])
            nd['connections_analysis'] = ca

        if char in profiles:
            nd['vocabulary_profile'] = profiles[char]
        else:
            nd['vocabulary_profile'] = {
                'total_words': 0, 'unique_lemmas': 0, 'ttr': None, 'mtld': None,
                'avg_word_length': None, 'data_reliability': 'none',
                'top_words': [], 'pos_distribution': {}, 'lemma_spot_check': [],
                'vocabulary_per_relationship': {},
                'note': 'character appears in network but has no recorded dialogue lines',
            }

        nodes.append(nd)

    edges = [
        {'source': u, 'target': v,
            'direct':     G[u][v].get('direct', 0),
            'copresence': G[u][v].get('copresence', 0),
            'words':      G[u][v].get('words', 0),
            'weight':     G[u][v].get('weight', 0)}
        for u, v in G.edges()
    ]

    data = {
        'nodes': nodes,
        'edges': edges,
        'metadata': {
            'total_nodes':       len(nodes),
            'total_edges':       len(edges),
            'total_communities': len(set(n['community'] for n in nodes)),
            'gender_stats': {
                'male':       sum(1 for n in nodes if n.get('gender') == 'M'),
                'female':     sum(1 for n in nodes if n.get('gender') == 'F'),
                'collective': sum(1 for n in nodes if n.get('gender') == 'C'),
                'unassigned': sum(1 for n in nodes if n.get('gender') is None),
            },
            'vocabulary_enrichment': {
                'nodes_enriched':           sum(1 for n in nodes if n['vocabulary_profile']['data_reliability'] != 'none'),
                'nodes_without_speech':     [n['id'] for n in nodes if n['vocabulary_profile']['data_reliability'] == 'none'],
                'reliability_thresholds':   {
                    'low':    f'< {RELIABILITY_LOW} words',
                    'medium': f'{RELIABILITY_LOW}–{RELIABILITY_MEDIUM} words',
                    'high':   f'>= {RELIABILITY_MEDIUM} words',
                },
            },
            'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        }
    }

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return data

result = enrich_and_export(G, character_metadata, connections_analysis, profiles, OUTPUT_FILE)

print(f"Saved → {OUTPUT_FILE}")
print(f"   {result['metadata']['total_nodes']} Nodes | {result['metadata']['total_edges']} Edges")
print(f"   {result['metadata']['total_communities']} Communities")
print(f"   Gender — M:{result['metadata']['gender_stats']['male']}  "
        f"F:{result['metadata']['gender_stats']['female']}  "
        f"C:{result['metadata']['gender_stats']['collective']}  "
        f"?:{result['metadata']['gender_stats']['unassigned']}")
ve = result['metadata']['vocabulary_enrichment']
print(f"   Vocabulary — {ve['nodes_enriched']} nodes enriched")
silent = ve['nodes_without_speech']
if silent:
    print(f"   Silent-only nodes: {', '.join(silent)}")
